In [1]:
OPENAI_API_KEY = "key_here"

In [7]:
from typing import TypedDict, Optional
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
import json
from pydantic import BaseModel
from langchain_core.output_parsers import PydanticOutputParser

In [14]:
import langchain
import requests
import importlib.metadata

print(f"LangChain version: {langchain.__version__}")
print(f"Requests version: {requests.__version__}")

try:
    l_openai_version = importlib.metadata.version("langchain-openai")
    print(f"LangChain-OpenAI version: {l_openai_version}")
except importlib.metadata.PackageNotFoundError:
    print("LangChain-OpenAI is not installed.")

LangChain version: 1.2.15
Requests version: 2.33.1
LangChain-OpenAI version: 1.2.1


In [8]:
# LLM
llm = ChatOpenAI(
    model="gpt-4o-mini",
    api_key=OPENAI_API_KEY,
    temperature=0
)

# STATE (MEMORY)
class LoanState(TypedDict):
    user_input: str
    principal: Optional[float]
    rate: Optional[float]
    time: Optional[float]
    emi: Optional[float]
    missing: Optional[bool]

# TOOL
@tool
def calculate_emi(principal: float, rate: float, time: float) -> float:
    """Calculate EMI"""
    print("...Running EMI tool")

    R = rate / 12 / 100
    N = time * 12

    emi = (principal * R * (1 + R)**N) / ((1 + R)**N - 1)
    return round(emi, 2)


In [9]:
# SCHEMA
class LoanExtraction(BaseModel):
    principal: float | None
    rate: float | None
    time: float | None

parser = PydanticOutputParser(pydantic_object=LoanExtraction)

# -------------------------
# NODE 1: EXTRACT (FIXED)
# -------------------------
def extract_node(state: LoanState) -> LoanState:

    prompt = f"""
    Extract principal, rate, and time from the input.

    Input: {state["user_input"]}

    If value is missing, return null.

    {parser.get_format_instructions()}
    """

    response = llm.invoke(prompt)

    # ✅ SAFE PARSING
    data = parser.parse(response.content)

    # 🔥 Merge with previous state (IMPORTANT)
    state["principal"] = data.principal or state.get("principal")
    state["rate"] = data.rate or state.get("rate")
    state["time"] = data.time or state.get("time")

    return state


In [10]:

# -------------------------
# NODE 2: VALIDATE
# -------------------------
def validate_node(state: LoanState) -> LoanState:
    if state["principal"] and state["rate"] and state["time"]:
        state["missing"] = False
    else:
        state["missing"] = True
    return state

# -------------------------
# NODE 3A: ASK USER
# -------------------------
def ask_node(state: LoanState) -> dict:
    missing = []
    if not state["principal"]:
        missing.append("principal")
    if not state["rate"]:
        missing.append("rate")
    if not state["time"]:
        missing.append("time")

    return {
        "message": f"Please provide missing values: {', '.join(missing)}"
    }

# -------------------------
# NODE 3B: CALCULATE
# -------------------------
def calculate_node(state: LoanState) -> LoanState:
    emi = calculate_emi.invoke({
        "principal": state["principal"],
        "rate": state["rate"],
        "time": state["time"]
    })

    state["emi"] = emi
    return state

# -------------------------
# NODE 4: OUTPUT
# -------------------------
def output_node(state: LoanState) -> dict:
    return {
        "principal": state["principal"],
        "rate": state["rate"],
        "time": state["time"],
        "emi": state["emi"]
    }

# -------------------------
# GRAPH BUILD
# -------------------------
builder = StateGraph(LoanState)

builder.add_node("extract", extract_node)
builder.add_node("validate", validate_node)
builder.add_node("ask_user", ask_node)
builder.add_node("calculate", calculate_node)
builder.add_node("output", output_node)

builder.set_entry_point("extract")
builder.add_edge("extract", "validate")

def route(state: LoanState):
    return "ask_user" if state["missing"] else "calculate"

builder.add_conditional_edges(
    "validate",
    route,
    {
        "ask_user": "ask_user",
        "calculate": "calculate"
    }
)

builder.add_edge("calculate", "output")
builder.add_edge("output", END)
builder.add_edge("ask_user", END)

# -------------------------
# MEMORY (IMPORTANT)
# -------------------------
memory = MemorySaver()

graph = builder.compile(checkpointer=memory)

# -------------------------
# CONFIG (THREAD ID = USER)
# -------------------------
config = {"configurable": {"thread_id": "user_1"}}

In [11]:
from IPython.display import display, Markdown

mermaid = graph.get_graph().draw_mermaid()

display(Markdown(f"```mermaid\n{mermaid}\n```"))

```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	extract(extract)
	validate(validate)
	ask_user(ask_user)
	calculate(calculate)
	output(output)
	__end__([<p>__end__</p>]):::last
	__start__ --> extract;
	calculate --> output;
	extract --> validate;
	validate -.-> ask_user;
	validate -.-> calculate;
	ask_user --> __end__;
	output --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```

In [12]:
# -------------------------
# TEST (YOUR EXACT CASE)
# -------------------------
print("\n--- First Query ---")
res1 = graph.invoke(
    {"user_input": "I took a loan of 2 lakh at 10% for 2 years. What is my EMI?"},
    config=config
)
print(res1)




--- First Query ---
...Running EMI tool
{'user_input': 'I took a loan of 2 lakh at 10% for 2 years. What is my EMI?', 'principal': 200000.0, 'rate': 10.0, 'time': 2.0, 'emi': 9228.99, 'missing': False}


In [13]:
print("\n--- Second Query ---")
res2 = graph.invoke(
    {"user_input": "Now if interest changes to 20%, then what is my EMI?"},
    config=config
)
print(res2)



--- Second Query ---
...Running EMI tool
{'user_input': 'Now if interest changes to 20%, then what is my EMI?', 'principal': 200000.0, 'rate': 20.0, 'time': 2.0, 'emi': 10179.16, 'missing': False}
